# 3. Crop Health Classification — MobileNetV2
3 classes: healthy / stressed / diseased.

In [ ]:
!pip -q install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224); BATCH = 32
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '/content/health/train', image_size=IMG_SIZE, batch_size=BATCH)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '/content/health/val', image_size=IMG_SIZE, batch_size=BATCH)
class_names = train_ds.class_names; print(class_names)

In [ ]:
aug = tf.keras.Sequential([
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.1),
    layers.RandomZoom(0.1), layers.Rescaling(1./127.5, offset=-1)])

base = MobileNetV2(input_shape=IMG_SIZE+(3,), include_top=False, weights='imagenet')
base.trainable = False
inputs = tf.keras.Input(shape=IMG_SIZE+(3,))
x = aug(inputs); x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.2)(x)
out = layers.Dense(3, activation='softmax')(x)
model = models.Model(inputs, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=30)

In [ ]:
# TFLite quantized export
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
open('crop_health.tflite', 'wb').write(conv.convert())